In [38]:
import os
import json
import requests
from tqdm import tqdm
from IPython.display import display, Markdown

In [39]:
'''The function searches Semantic Scholar for research papers on a topic, returns selected paper details, and gracefully handles rate limits by retrying after waiting.'''
import time
import requests

def fetch_papers(topic, limit=3, retries=3, wait_seconds=5):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"

    params = {
        "query": topic,
        "limit": limit,
        "fields": "title,authors,year,abstract,url"
    }

    headers = {
        "User-Agent": "ResearchPaperFetcher/1.0"
    }

    for attempt in range(retries):
        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=10
            )

            if response.status_code == 200:
                return response.json().get("data", [])

            elif response.status_code == 429:
                wait = wait_seconds * (2 ** attempt)
                print(f"Rate limit hit. Retrying in {wait} seconds...")
                time.sleep(wait)

            else:
                response.raise_for_status()

        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            break

    print("Could not fetch papers due to repeated failures.")
    return []



In [40]:
'''This function iterates through a list of research papers and prints each paper’s title, authors, year, abstract, and link in a clear, formatted way.'''
def show_papers(papers):
    for i, paper in enumerate(papers, start=1):
        authors = ", ".join(a["name"] for a in paper.get("authors", []))

        print(f"\nPaper {i}")
        print("Title:", paper.get("title"))
        print("Authors:", authors)
        print("Year:", paper.get("year"))
        print("\nAbstract:")
        print(paper.get("abstract"))
        print("\nPaper Link:", paper.get("url"))
        print("-" * 60)



In [41]:
'''This code searches for research papers related to the topic “deepfake” using fetch_papers and then displays the retrieved papers using show_papers.'''
topic = "fake news detection"

print("Fetching papers...")
papers = fetch_papers(topic)

show_papers(papers)


Fetching papers...

Paper 1
Title: Fake News Detection on Social Media: A Data Mining Perspective
Authors: Kai Shu, A. Sliva, Suhang Wang, Jiliang Tang, Huan Liu
Year: 2017

Abstract:
Social media for news consumption is a double-edged sword. On the one hand, its low cost, easy access, and rapid dissemination of information lead people to seek out and consume news from social media. On the other hand, it enables the wide spread of \fake news", i.e., low quality news with intentionally false information. The extensive spread of fake news has the potential for extremely negative impacts on individuals and society. Therefore, fake news detection on social media has recently become an emerging research that is attracting tremendous attention. Fake news detection on social media presents unique characteristics and challenges that make existing detection algorithms from traditional news media ine ective or not applicable. First, fake news is intentionally written to mislead readers to believ

In [42]:
'''The function transforms raw API paper data into a clean, simplified dataset ready for analysis or saving to JSON.'''
import json

def prepare_dataset(papers):
    dataset = []

    for paper in papers:
        record = {
            "title": paper.get("title"),
            "authors": [a["name"] for a in paper.get("authors", [])],
            "year": paper.get("year"),
            "abstract": paper.get("abstract"),
            "paper_url": paper.get("url")
        }

        dataset.append(record)

    return dataset


In [43]:
'''This code converts the fetched papers into a structured dataset using prepare_dataset and then prints how many papers are in that dataset.'''
dataset = prepare_dataset(papers)

print("Number of papers in dataset:", len(dataset))


Number of papers in dataset: 3


In [44]:
'''This code loops through the prepared dataset and prints key details of each paper in a clean, numbered format.'''
for i, d in enumerate(dataset, start=1):
    print("Paper", i)
    print("Title:", d["title"])
    print("Authors:", ", ".join(d["authors"]))
    print("Year:", d["year"])
    print("URL:", d["paper_url"])
    print("-" * 50)

Paper 1
Title: Fake News Detection on Social Media: A Data Mining Perspective
Authors: Kai Shu, A. Sliva, Suhang Wang, Jiliang Tang, Huan Liu
Year: 2017
URL: https://www.semanticscholar.org/paper/cb40a5e6d4fc0290452345791bb91040aed76961
--------------------------------------------------
Paper 2
Title: “Liar, Liar Pants on Fire”: A New Benchmark Dataset for Fake News Detection
Authors: William Yang Wang
Year: 2017
URL: https://www.semanticscholar.org/paper/03c294ad75bd1bac92217419ac25358227f6a901
--------------------------------------------------
Paper 3
Title: Advancing Fake News Detection: Hybrid Deep Learning With FastText and Explainable AI
Authors: Ehtesham Hashmi, Sule YAYILGAN YILDIRIM, M. Yamin, Subhan Ali, Mohamed Abomhara
Year: 2024
URL: https://www.semanticscholar.org/paper/a81ed24cbdbd77fbe399e0556cae841728a90c51
--------------------------------------------------


In [45]:
'''This function retrieves the list of papers cited by a given paper and returns their basic details.'''
def fetch_references(paper_id, limit=5):
    url = f"https://api.semanticscholar.org/graph/v1/paper/{paper_id}/references"

    params = {
        "limit": limit,
        "fields": "title,authors,year,url"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("Failed to fetch references")
        return []

    data = response.json().get("data")

    if not data:
        print("No references found for this paper.")
        return []

    references = []
    for r in data:
        cited = r.get("citedPaper")
        if cited:
            references.append(cited)

    return references

In [46]:
'''it neatly displays all reference papers (cited works) on the console.'''
def show_references(references):
    for i, ref in enumerate(references, start=1):
        authors = ", ".join(a["name"] for a in ref.get("authors", []))

        print("Reference", i)
        print("Title:", ref.get("title"))
        print("Authors:", authors)
        print("Year:", ref.get("year"))
        print("Link:", ref.get("url"))
        print("-" * 50)


In [47]:
'''This code selects the first fetched paper, retrieves the papers it cites using fetch_references, and then displays those references using show_references.'''
paper_id = papers[0]["paperId"]   # pick first fetched paper

print("Fetching references...")
references = fetch_references(paper_id)

show_references(references)


Fetching references...
Reference 1
Title: Clickbait Detection
Authors: Suhaib Khater, Oraib Al-Sahlee, Daoud M. Daoud, M. S. A. El-Seoud
Year: 2018
Link: https://www.semanticscholar.org/paper/ec68cf304903671999e791c39115bc3ec08335ed
--------------------------------------------------
Reference 2
Title: Attributed Signed Network Embedding
Authors: Suhang Wang, C. Aggarwal, Jiliang Tang, Huan Liu
Year: 2017
Link: https://www.semanticscholar.org/paper/2e8cb18e9dc2cec1ef25d08e4567a2840692a5a0
--------------------------------------------------
Reference 3
Title: Rumors: Uses, Interpretation and Necessity
Authors: J. Kapferer
Year: 2017
Link: https://www.semanticscholar.org/paper/ad6956d28706ff7bbb29e9036d39a2126e7c6888
--------------------------------------------------
Reference 4
Title: Adaptive Spammer Detection with Sparse Group Modeling
Authors: Liang Wu, Xia Hu, Fred Morstatter, Huan Liu
Year: 2017
Link: https://www.semanticscholar.org/paper/ca7fbbfb4f701df71c0e2b4aedb4f9b1d1574aac
----

In [48]:
paper_id

'cb40a5e6d4fc0290452345791bb91040aed76961'